# Advisory parameter selection

These routines benchmark suggestions for one problem; they do not change `UniformFmm` defaults. Performance depends on geometry, particle count, target distribution, CPU, GPU, thread count, backend, and compiler optimisation. Accuracy additionally depends on moments, targets, order, depth, and deterministic sample selection.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cdfmm

particle_counts = [1000, 5000, 10000, 30000]
performance_orders = [2, 4, 6, 8]
candidate_depths = [2, 3, 4]
repetitions = 3

rng = np.random.default_rng(13)
maximum_particles = max(particle_counts)
sources = rng.uniform(-1, 1, (maximum_particles, 3))
targets = rng.uniform(-1, 1, (maximum_particles, 3))
moments = rng.normal(size=(maximum_particles, 3))


## Performance sweep

The CPU backend is sequential, so measured total time selects the candidate; balance is diagnostic. Partial CUDA overlaps near and far work and uses `max(T_near, T_far)` as its branch-cost heuristic while retaining wall time. The sweep uses nested prefixes of one deterministic geometry so changes with particle count are not confounded by unrelated random samples.

At depth 1, all eight leaves are mutual neighbours and the complete interaction is evaluated through P2P, making this effectively direct all-to-all with additional tree overhead. At depth 2, separated leaves exist: nearby leaf pairs use P2P while well-separated pairs use M2L, so depth 2 is already a genuine FMM configuration for a spatially distributed problem.


In [ ]:
performance_tables = []
performance_suggestions = {}

for particle_count in particle_counts:
    source_subset = sources[:particle_count]
    target_subset = targets[:particle_count]
    moment_subset = moments[:particle_count]

    for order in performance_orders:
        suggestion = cdfmm.suggest_depth_for_performance(
            source_subset, target_subset, moment_subset, order=order,
            backend=cdfmm.ExecutionBackend.CPU_STATIC,
            candidate_depths=candidate_depths, repetitions=repetitions,
        )
        performance_suggestions[(particle_count, order)] = suggestion

        candidates = pd.DataFrame(suggestion["candidates"])
        candidates.insert(0, "order", order)
        candidates.insert(0, "particle_count", particle_count)
        candidates["suggested_depth"] = suggestion["suggested_depth"]
        performance_tables.append(candidates)

timing_table = pd.concat(performance_tables, ignore_index=True)
display(timing_table[["particle_count", "order", "depth", "status",
                      "near_seconds", "far_seconds", "balance_ratio",
                      "evaluation_seconds", "suggested_depth"]])

selected_timings = timing_table[
    timing_table.depth == timing_table.suggested_depth
]["particle_count order suggested_depth evaluation_seconds".split()]
display(selected_timings.pivot(index="particle_count", columns="order",
                               values="suggested_depth"))

figure, axes = plt.subplots(
    len(particle_counts), len(performance_orders),
    figsize=(4.2 * len(performance_orders), 3.2 * len(particle_counts)),
    sharex=True, squeeze=False,
)
for row, particle_count in enumerate(particle_counts):
    for column, order in enumerate(performance_orders):
        axis = axes[row, column]
        candidates = timing_table[
            (timing_table.particle_count == particle_count)
            & (timing_table.order == order)
            & (timing_table.status == "ok")
        ].sort_values("depth")
        axis.plot(candidates.depth, candidates.near_seconds, "o-", label="near")
        axis.plot(candidates.depth, candidates.far_seconds, "o-", label="far")
        axis.plot(candidates.depth, candidates.evaluation_seconds, "o-", label="total")
        suggested_depth = performance_suggestions[(particle_count, order)]["suggested_depth"]
        if suggested_depth >= 0:
            axis.axvline(suggested_depth, color="black", linestyle="--",
                         label="suggested")
        axis.set_title(f"N={particle_count:,}, order={order}")
        axis.set_xticks(candidate_depths)
        axis.set_yscale("log")
        axis.grid(alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=len(labels))
figure.supxlabel("tree depth")
figure.supylabel("seconds per evaluation (logarithmic scale)")
figure.suptitle("Near-field, far-field, and total evaluation time", y=1.02)
figure.tight_layout();


## Sampled accuracy sweep

The direct reference is computed once for the same deterministic targets. The recommendation is the fastest tested pair meeting the requested sampled RMS relative field error, not the most accurate pair.


In [ ]:
accuracy = cdfmm.suggest_parameters_for_accuracy(
    sources, targets, moments, desired_accuracy=1e-3,
    candidate_orders=[2, 4, 6, 8], candidate_depths=[2, 3, 4],
    sample_size=128, repetitions=3,
)
accuracy_table = pd.DataFrame(accuracy["candidates"])
error_heatmap = accuracy_table.pivot(index="depth", columns="order",
                                     values="rms_relative_error")
display(error_heatmap)
display(accuracy_table[["depth", "order", "rms_relative_error",
                        "evaluation_seconds", "satisfies_accuracy"]])

fig, axis = plt.subplots()
image = axis.imshow(error_heatmap, origin="lower", aspect="auto")
axis.set_xticks(range(len(error_heatmap.columns)), error_heatmap.columns)
axis.set_yticks(range(len(error_heatmap.index)), error_heatmap.index)
axis.set(xlabel="order", ylabel="depth", title="Sampled RMS relative field error")
if accuracy["suggested_order"] >= 0:
    x = list(error_heatmap.columns).index(accuracy["suggested_order"])
    y = list(error_heatmap.index).index(accuracy["suggested_depth"])
    axis.plot(x, y, "wx", markersize=14, markeredgewidth=3)
fig.colorbar(image, ax=axis);


For repeated static problems, run an adviser once during setup, then explicitly assign its returned values to `options.expansion_order` and `options.tree.max_level`. A sampled estimate is not a rigorous global error bound or a guarantee of universal optimality.
